# Magi sequence viewer — Magi OCR only
Set a split and sequence ID, then run the cells in order. Use the page dropdown and checkboxes to inspect the Magi output, and the text dropdown to inspect a crop and its full OCR.

Uses original Magi detections, character clusters, associations, and OCR. No Paddle OCR, OCR selection, character cleanup, GT, or role classifier is used. All text detections remain visible; red means Magi essential and gray means non-essential.

Use your existing Magi kernel with CUDA PyTorch, Transformers, Pillow, numpy, pandas, matplotlib, and ipywidgets. Restart the kernel first if Qwen or MangaLMM still occupies the GPU.

In [1]:
from pathlib import Path
import hashlib
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from PIL import Image
from IPython.display import display
from transformers import AutoModel

MAGI_ROOT = Path(r"C:\Users\ESHAAN\BYTE\Applied-Task\Magi")
MODEL_PATH = Path(r"C:\MODELS\magiv2")
SPLIT = "development"
SEQUENCE_ID = "seq_2fae297a18af2a57"
USE_CACHE = True  # False runs Magi again, including its own OCR.

CACHE_DIR = MAGI_ROOT / "magi_sequence_viewer_cache" / SPLIT / SEQUENCE_ID
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def json_ready(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {key: json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    return value

In [2]:
sequence_root = MAGI_ROOT / "TaskData" / SPLIT / "images" / SEQUENCE_ID
assert sequence_root.is_dir(), f"Missing sequence folder: {sequence_root}"
PAGE_FILES = sorted(sequence_root.glob("*.png"))
assert len(PAGE_FILES) == 3, f"Expected three pages, found {len(PAGE_FILES)}"
chapter_pages = []
page_info = []
for path in PAGE_FILES:
    with Image.open(path) as image:
        image = image.convert("L").convert("RGB")  # same preprocessing as the pipeline
        chapter_pages.append(np.array(image))
        page_info.append({"name": path.name, "sha256": sha256(path),
                          "width": image.width, "height": image.height})
model_files = [path for path in MODEL_PATH.iterdir() if path.is_file()]
inputs = {"sequence_id": SEQUENCE_ID, "split": SPLIT, "pages": page_info,
          "preprocessing": "grayscale_then_RGB", "model_path": str(MODEL_PATH),
          "model_files": [{"name": path.name, "size": path.stat().st_size,
                           "mtime_ns": path.stat().st_mtime_ns}
                          for path in sorted(model_files)]}
print("Sequence:", SEQUENCE_ID, "Split:", SPLIT)
print("Pages:", [path.name for path in PAGE_FILES])

Sequence: seq_2fae297a18af2a57 Split: development
Pages: ['01.png', '02.png', '03.png']


## Get the raw Magi output
Cache reuse always reads `magi_raw.json`, whose `ocr` field is Magi OCR. It never reads `ocr_selection.json` or replaces OCR with another source. Existing pipeline caches must match the sequence, split, original page hashes, and model path. Viewer results are saved separately.

In [3]:
viewer_cache = CACHE_DIR / "magi_raw.json"
results = None
source = None
if USE_CACHE and viewer_cache.exists():
    cached = read_json(viewer_cache)
    if cached.get("inputs") == inputs:
        results = cached["data"]
        source = str(viewer_cache)

if USE_CACHE and results is None:
    # Try the existing pipeline's raw cache; no derived/selected OCR is used.
    for raw_path in sorted((MAGI_ROOT / "paddleOCR-Magi").rglob("magi_raw.json")):
        if raw_path.parent.name != SEQUENCE_ID:
            continue
        manifest_path = raw_path.parent / "manifest.json"
        if not manifest_path.exists():
            continue
        manifest = read_json(manifest_path)
        cached_inputs = manifest.get("inputs", {})
        if (cached_inputs.get("sequence_id") == SEQUENCE_ID
            and cached_inputs.get("split") == SPLIT
            and cached_inputs.get("pages") == page_info
            and cached_inputs.get("preprocessing") == "grayscale_then_RGB"
            and manifest.get("model", {}).get("path") == str(MODEL_PATH)):
            results = read_json(raw_path)["data"]
            source = str(raw_path)
            break

if results is None:
    assert torch.cuda.is_available(), "Use a CUDA-enabled Magi kernel."
    magi_model = AutoModel.from_pretrained(
        str(MODEL_PATH), trust_remote_code=True, local_files_only=True
    ).cuda().eval()
    with torch.inference_mode():
        results = magi_model.do_chapter_wide_prediction(
            chapter_pages, {"images": [], "names": []}, use_tqdm=True, do_ocr=True
        )
    results = json_ready(results)
    source = "Fresh Magi inference (do_ocr=True)"

assert len(results) == len(chapter_pages)
for result in results:
    assert len(result["texts"]) == len(result["ocr"]), "Text/OCR mismatch"
    assert len(result["characters"]) == len(result["character_cluster_labels"])
viewer_cache.write_text(json.dumps({"inputs": inputs, "source": source, "data": results},
                                   indent=2, ensure_ascii=False), encoding="utf-8")
print("Source:", source)
print("Saved:", viewer_cache)
display(pd.DataFrame([{"page": i + 1, "texts": len(r["texts"]),
                       "characters": len(r["characters"]), "panels": len(r.get("panels", []))}
                      for i, r in enumerate(results)]))

Source: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\magi_sequence_viewer_cache\development\seq_2fae297a18af2a57\magi_raw.json
Saved: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\magi_sequence_viewer_cache\development\seq_2fae297a18af2a57\magi_raw.json


,page,texts,characters,panels
0,1,11,7,6
1,2,9,7,5
2,3,8,8,6


## Page viewer
The controls follow the reference pipeline viewer. Cluster colors and links refer to native Magi cluster labels, without a separate entity-association stage. OCR labels on the image are shortened for readability; full OCR appears in the table below it.

In [4]:
def center(box):
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2, (y1 + y2) / 2)

def draw_box(ax, box, color, label=""):
    x1, y1, x2, y2 = box
    ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                   linewidth=2, edgecolor=color, facecolor="none"))
    if label:
        ax.text(x1, max(0, y1 - 4), label, fontsize=8, color="white",
                bbox=dict(facecolor="black", alpha=0.7, edgecolor="none"))

def show_page(page_index=0, panels=False, texts=True, characters=True, tails=False,
              same_character=False, text_character=True, text_tail=False,
              ocr=True, ids=True, cluster_colors=True, save_path=None):
    result = results[page_index]
    labels = result["character_cluster_labels"]
    cmap = plt.get_cmap("tab20")
    def char_color(index):
        label = int(labels[index])
        return cmap(label % 20) if cluster_colors and label >= 0 else "blue"

    fig, ax = plt.subplots(figsize=(12, 18))
    ax.imshow(chapter_pages[page_index])
    if panels:
        for i, box in enumerate(result.get("panels", [])):
            draw_box(ax, box, "green", f"PN{i+1}" if ids else "")
    if texts:
        for i, box in enumerate(result["texts"]):
            essential = result.get("is_essential_text", [])
            color = "red" if i >= len(essential) or essential[i] else "gray"
            parts = [f"PG{page_index+1}:T{i+1}"] if ids else []
            if ocr:
                text = result["ocr"][i] or ""
                parts.append(text[:62] + "..." if len(text) > 65 else text)
            draw_box(ax, box, color, " | ".join(parts))
    if characters:
        for i, box in enumerate(result["characters"]):
            cluster = f"CL{int(labels[i])+1}" if int(labels[i]) >= 0 else "unassigned"
            label = f"C{i+1} / {cluster}" if ids else ""
            draw_box(ax, box, char_color(i), label)
    if tails:
        for i, box in enumerate(result.get("tails", [])):
            draw_box(ax, box, "purple", f"TAIL{i+1}" if ids else "")
    if same_character:
        groups = {}
        for i, label in enumerate(labels):
            if int(label) >= 0:
                groups.setdefault(int(label), []).append(i)
        for indices in groups.values():
            for index in indices[1:]:
                a = center(result["characters"][indices[0]])
                b = center(result["characters"][index])
                ax.plot([a[0], b[0]], [a[1], b[1]], color=char_color(index), linewidth=1.5)
    for enabled, key, target, style in [
        (text_character, "text_character_associations", "characters", "--"),
        (text_tail, "text_tail_associations", "tails", ":"),
    ]:
        if enabled:
            for text_index, target_index in result.get(key, []):
                if not (0 <= text_index < len(result["texts"])
                        and 0 <= target_index < len(result.get(target, []))):
                    continue
                a = center(result["texts"][text_index])
                b = center(result[target][target_index])
                color = char_color(target_index) if target == "characters" else "purple"
                ax.plot([a[0], b[0]], [a[1], b[1]], color=color, linestyle=style, linewidth=1.5)
                if ids:
                    target_id = f"C{target_index+1}" if target == "characters" else f"TAIL{target_index+1}"
                    ax.text((a[0]+b[0])/2, (a[1]+b[1])/2, f"T{text_index+1} -> {target_id}", fontsize=7)
    ax.set_title(f"{SPLIT} | {SEQUENCE_ID} | PG{page_index+1}: {PAGE_FILES[page_index].name} | Magi OCR")
    ax.axis("off")
    fig.tight_layout()
    if save_path is not None:
        path = Path(save_path)
        path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(path, bbox_inches="tight", dpi=150)
    plt.show()
    plt.close(fig)

    essential = result.get("is_essential_text", [])
    display(pd.DataFrame([{"ID": f"PG{page_index+1}:T{i+1}", "Magi OCR": text,
                           "Magi essential": essential[i] if i < len(essential) else None,
                           "characters": ", ".join(f"C{c+1}" for t, c in result.get("text_character_associations", []) if t == i)}
                          for i, text in enumerate(result["ocr"])],
                         columns=["ID", "Magi OCR", "Magi essential", "characters"]))

In [5]:
page_control = widgets.Dropdown(
    options=[(f"PG{i+1} - {path.name}", i) for i, path in enumerate(PAGE_FILES)], description="Page:"
)
flags = {name: widgets.Checkbox(value=value, description=name.replace("_", " "), indent=False)
         for name, value in dict(panels=False, texts=True, characters=True, tails=False,
                                 same_character=False, text_character=True, text_tail=False,
                                 ocr=True, ids=True, cluster_colors=True).items()}
viewer_output = widgets.interactive_output(show_page, dict(page_index=page_control, **flags))
display(widgets.VBox([page_control,
                      widgets.HBox(list(flags.values())[:5]),
                      widgets.HBox(list(flags.values())[5:])]), viewer_output)

Output()

## Inspect a text crop and full Magi OCR
Choose a page, then its text ID. This uses the original Magi box and OCR index.

In [6]:
def inspect_text(page_index=0, text_index=0):
    result = results[page_index]
    if not result["texts"]:
        print("No text detections on this page.")
        return
    box = result["texts"][text_index]
    image = chapter_pages[page_index]
    height, width = image.shape[:2]
    x1, y1, x2, y2 = box
    x1, y1 = max(0, int(np.floor(x1))), max(0, int(np.floor(y1)))
    x2, y2 = min(width, int(np.ceil(x2))), min(height, int(np.ceil(y2)))
    if x2 > x1 and y2 > y1:
        display(Image.fromarray(image[y1:y2, x1:x2]))
    print(f"PG{page_index+1}:T{text_index+1}")
    print("Magi OCR:", result["ocr"][text_index])
    print("Box:", box)
    print("Associated characters:", [f"C{c+1}" for t, c in result.get("text_character_associations", []) if t == text_index])

crop_page = widgets.Dropdown(options=page_control.options, description="Page:")
crop_text = widgets.Dropdown(description="Text:")
crop_output = widgets.Output()

def show_crop(change=None):
    with crop_output:
        crop_output.clear_output(wait=True)
        if crop_text.value is not None:
            inspect_text(crop_page.value, crop_text.value)
        else:
            print("No text detections on this page.")

def update_text_options(change=None):
    count = len(results[crop_page.value]["texts"])
    crop_text.options = [(f"PG{crop_page.value+1}:T{i+1}", i) for i in range(count)]
    crop_text.value = 0 if count else None
    show_crop()

crop_page.observe(update_text_options, names="value")
crop_text.observe(show_crop, names="value")
update_text_options()
display(widgets.HBox([crop_page, crop_text]), crop_output)

Output()

## Direct calls if widgets are unavailable
Uncomment the calls you want. The `save_path` option exports the displayed annotation.

In [7]:
# show_page(0)
# show_page(1, panels=True, tails=True, same_character=True, text_tail=True)
# inspect_text(page_index=0, text_index=0)
# show_page(0, save_path=CACHE_DIR / "annotations" / "page_1.png")